# Member 2 - Pipeline A: raw frames (CNN + LSTM)
Pretrained CNN on every frame -> (Bi)LSTM -> classifier (LRCN). Needs `processed/frames/` and
`processed/splits/` from Member 1 (shared Drive folder or Kaggle dataset).

**Use a GPU runtime** (Runtime -> Change runtime type -> T4). Training resumes automatically after a disconnect.

## 0. Setup (run every new session)
Set `BRANCH` to the branch you are working on.

In [ ]:
REPO = "https://github.com/Rani553/indian-sign-language-video-recognition.git"
BRANCH = "feature/cnn-lstm"   # use "develop" once this work is merged
import os
if not os.path.exists("indian-sign-language-video-recognition"):
    !git clone -q $REPO
%cd indian-sign-language-video-recognition
!git fetch -q && git checkout -q $BRANCH && git pull -q
!git log --oneline -3

In [ ]:
# Colab: mount Drive so data and checkpoints survive disconnects. Kaggle: skip (this just prints).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab")

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
!bash {"scripts/setup_kaggle.sh" if ON_KAGGLE else "scripts/setup_colab.sh"}

## 1. Check the data
On Kaggle, attach the processed dataset and add `paths.processed_dir=/kaggle/input/<dataset>` to every `--set` below.

In [ ]:
import os, glob
from src.utils.config import load_config, resolve_paths
paths = resolve_paths(load_config("configs/cnn_lstm.yaml"))
for k in ("splits_dir", "frames_dir"):
    print(k, paths[k], "| exists:", paths[k].exists())
print("cached videos:", len(glob.glob(str(paths["frames_dir"] / "*.npy"))))

## 2. Quick check (2 epochs, tiny subset)
Run this first. If it works, the full run will too.

In [ ]:
!python -m src.training.train_cnn_lstm --config configs/cnn_lstm.yaml --tiny

## 3. Main model
Baseline: MobileNetV2, 16 frames, CNN frozen for 3 epochs and then fine-tuned.

In [ ]:
!python -m src.training.train_cnn_lstm --config configs/cnn_lstm.yaml

In [ ]:
from IPython.display import Image
Image(str(paths["results_root"] / "cnn_lstm_mobilenet_v2" / "curves.png"))

## 4. Ablations
One change at a time (backbone, 8 vs 16 frames, frozen vs fine-tuned, pooling). Each run resumes, so re-run this cell after a disconnect.
Pick groups: `!bash scripts/run_ablations_cnn.sh backbone frames`

In [ ]:
!bash scripts/run_ablations_cnn.sh

## 5. Collect the results
Best validation accuracy of every run (history.csv). The test-set table is made by Member 4's `evaluate` / `compare`.

In [ ]:
import pandas as pd
rows = []
for h in sorted(paths["results_root"].glob("cnn_lstm_*/history.csv")):
    d = pd.read_csv(h)
    rows.append({"experiment": h.parent.name, "best_val_acc": d["val_acc"].max(),
                 "best_epoch": int(d["val_acc"].idxmax()), "epochs_run": len(d),
                 "sec_per_epoch": d["epoch_time_s"].mean()})
pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).round(4)

## 6. Share the checkpoint
Upload `results/<experiment>/best.pt` (and `config.yaml`) to the shared Drive/Kaggle folder. **Do not commit `.pt` files to Git.**